In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

In [2]:
df = pd.read_csv("quote_data.csv")

In [3]:
df.head()

,quote,Author
0,“The world as we have created it is a process ...,Albert Einstein
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling
2,“There are only two ways to live your life. On...,Albert Einstein
3,"“The person, be it gentleman or lady, who has ...",Jane Austen
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe


In [4]:
df.shape

(3038, 2)

In [5]:
quotes = df['quote']
quotes.head()

,quote
0,“The world as we have created it is a process ...
1,"“It is our choices, Harry, that show what we t..."
2,“There are only two ways to live your life. On...
3,"“The person, be it gentleman or lady, who has ..."
4,"“Imperfection is beauty, madness is genius and..."


In [6]:
quotes = quotes.str.lower()

In [7]:
import string
translator = str.maketrans('', '', string.punctuation)
quotes = quotes.apply(lambda x: x.translate(translator))

In [8]:
quotes.head()

,quote
0,“the world as we have created it is a process ...
1,“it is our choices harry that show what we tru...
2,“there are only two ways to live your life one...
3,“the person be it gentleman or lady who has no...
4,“imperfection is beauty madness is genius and ...


In [9]:
from tensorflow.keras.preprocessing.text import Tokenizer

In [10]:
vocab_size = 10000

tokinizer = Tokenizer(num_words=vocab_size)
tokinizer.fit_on_texts(quotes)

In [11]:
word_index = tokinizer.word_index
print(len(word_index))
list(word_index.items())[:10]

8978


[('the', 1),
 ('you', 2),
 ('to', 3),
 ('and', 4),
 ('a', 5),
 ('i', 6),
 ('is', 7),
 ('of', 8),
 ('that', 9),
 ('it', 10)]

In [12]:
sequence = tokinizer.texts_to_sequences(quotes)

In [13]:
for i in range(3):
  print(quotes[i])

“the world as we have created it is a process of our thinking it cannot be changed without changing our thinking”
“it is our choices harry that show what we truly are far more than our abilities”
“there are only two ways to live your life one is as though nothing is a miracle the other is as though everything is a miracle”


In [14]:
for i in range(3):
  print(sequence[i])

[713, 62, 29, 19, 16, 946, 10, 7, 5, 1156, 8, 70, 293, 10, 145, 12, 809, 104, 752, 70, 2461]
[947, 7, 70, 871, 373, 9, 433, 21, 19, 465, 14, 294, 52, 54, 70, 3676]
[1337, 14, 53, 201, 714, 3, 81, 15, 36, 37, 7, 29, 329, 93, 7, 5, 1157, 1, 101, 7, 29, 329, 126, 7, 5, 3677]


In [15]:
X = []
y = []

for seq in sequence:
  for i in range(1,len(seq)):
    input_seq = seq[:i]
    output_seq = seq[i]
    X.append(input_seq)
    y.append(output_seq)


In [16]:
len(X)

85271

In [17]:
len(y)

85271

In [18]:
max_len = max(len(x) for x in X)
print(max_len)

745


In [19]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
X_padded = pad_sequences(X, maxlen=max_len, padding='pre')


In [20]:
y = np.array(y)

In [21]:
X_padded.shape

(85271, 745)

In [22]:
from tensorflow.keras.utils import to_categorical
y_one_hot = to_categorical(y, num_classes=vocab_size)

In [23]:
y.shape

(85271,)

In [24]:
y_one_hot.shape

(85271, 10000)

In [25]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding,SimpleRNN,LSTM, Dense

In [26]:
embedding_dim = 50
rnn_units = 128

In [27]:
rnn_model = Sequential()

rnn_model.add(
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_len)
)
rnn_model.add(SimpleRNN(units=rnn_units))
rnn_model.add(Dense(units=vocab_size, activation='softmax'))

In [28]:
rnn_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [29]:
rnn_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [30]:
lstm_model = Sequential()
lstm_model.add(
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_len)
)
lstm_model.add(LSTM(units=rnn_units))
lstm_model.add(Dense(units=vocab_size, activation='softmax'))

In [31]:
lstm_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [32]:
lstm_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [34]:
epochs = 10
batch_size = 128
history_rnn = rnn_model.fit(X_padded,
                            y_one_hot,
                            epochs=epochs,
                            batch_size=batch_size,
                            validation_split=0.1)

Epoch 1/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 46s 70ms/step - accuracy: 0.0451 - loss: 6.7102 - val_accuracy: 0.0550 - val_loss: 6.5088
Epoch 2/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 37s 62ms/step - accuracy: 0.0802 - loss: 6.0654 - val_accuracy: 0.0899 - val_loss: 6.2881
Epoch 3/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 37s 62ms/step - accuracy: 0.1036 - loss: 5.6951 - val_accuracy: 0.1021 - val_loss: 6.2797
Epoch 4/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 37s 61ms/step - accuracy: 0.1212 - loss: 5.3888 - val_accuracy: 0.1099 - val_loss: 6.2830
Epoch 5/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 37s 61ms/step - accuracy: 0.1362 - loss: 5.1135 - val_accuracy: 0.1133 - val_loss: 6.3555
Epoch 6/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 37s 61ms/step - accuracy: 0.1500 - loss: 4.8625 - val_accuracy: 0.1114 - val_loss: 6.3860
Epoch 7/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 37s 61ms/step - accuracy: 0.1683 - loss: 4.6311 - val_accuracy: 0.1112 - val_loss: 6.4717
Epoch 8/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 37s 61ms/step - accuracy: 0.1868 - loss: 4.4171 - 

In [35]:
epochs = 10
batch_size = 128
history_lstm = lstm_model.fit(X_padded,
                            y_one_hot,
                            epochs=epochs,
                            batch_size=batch_size,
                            validation_split=0.1)

Epoch 1/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 38s 54ms/step - accuracy: 0.0397 - loss: 6.7573 - val_accuracy: 0.0455 - val_loss: 6.6729
Epoch 2/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 31s 52ms/step - accuracy: 0.0584 - loss: 6.3101 - val_accuracy: 0.0661 - val_loss: 6.5556
Epoch 3/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 31s 52ms/step - accuracy: 0.0809 - loss: 6.0333 - val_accuracy: 0.0906 - val_loss: 6.4397
Epoch 4/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 32s 53ms/step - accuracy: 0.0985 - loss: 5.8074 - val_accuracy: 0.0972 - val_loss: 6.4078
Epoch 5/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 41s 53ms/step - accuracy: 0.1104 - loss: 5.6171 - val_accuracy: 0.1038 - val_loss: 6.3991
Epoch 6/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 41s 53ms/step - accuracy: 0.1195 - loss: 5.4395 - val_accuracy: 0.1060 - val_loss: 6.3942
Epoch 7/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 33s 55ms/step - accuracy: 0.1298 - loss: 5.2754 - val_accuracy: 0.1095 - val_loss: 6.4347
Epoch 8/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 32s 53ms/step - accuracy: 0.1377 - loss: 5.1224 - 

In [39]:
lstm_model.save("lstm_model.h5")

In [40]:
index_to_word = {}
for word, index in word_index.items():
  index_to_word[index] = word

In [41]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [42]:
def predictor(model,tokenizer,text,max_len):
  text = text.lower()

  seq = tokenizer.texts_to_sequences([text])[0]
  seq = pad_sequences([seq], maxlen=max_len, padding='pre')

  pred = model.predict(seq,verbose = 0)
  pred_index = np.argmax(pred)
  return index_to_word[pred_index]


In [43]:
seed_text = "what are you"
next_word = predictor(lstm_model,tokinizer,seed_text,max_len)
print(next_word)

can


In [44]:
def generate_text(model,tokenizer,seed_text,max_len,n_words):
  for _ in range(n_words):
    next_word = predictor(model,tokenizer,seed_text,max_len)
    if next_word == "":
      break
    seed_text += " " + next_word
  return seed_text

In [45]:
seed = "are you a "
generate_text = generate_text(lstm_model,tokinizer,seed,max_len,10)
print(generate_text)

are you a  book you can be a book and the world you


In [46]:
import pickle
with open("tokenizer.pkl", "wb") as f:
  pickle.dump(tokinizer, f)

In [47]:
with open("max_len.pkl", "wb") as f:
  pickle.dump(max_len, f)